# 07 SOKE Gemma3 R256 A512 Train

Train the selected LoRA setup from the rank sweep: `r=256`, `alpha=512`, `alpha/r=2.0`. The run does 30 epochs of `lm_pretrain`, then 5 epochs of `lm_instruct` initialized from the pretrain adapter. It is designed for Colab: Drive artifacts are copied to local storage first, checkpoints/logs are synced back to Drive every epoch, and stages can resume after runtime resets.


In [ ]:
# Run configuration: edit this cell first in Colab.
from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess
import sys

IN_COLAB = Path('/content').exists()
DEFAULT_DRIVE_BUNDLE_ROOT = Path('/content/drive/MyDrive/folder/COLAB/Tokenizer/04_soke_gemma3_270m_lora_lm')
DEFAULT_LOCAL_BUNDLE_ROOT = Path('/content/04_soke_gemma3_270m_lora_lm') if IN_COLAB else Path.cwd()

DRIVE_BUNDLE_ROOT = Path(os.environ.get('SOKE_GEMMA_R256_DRIVE_BUNDLE_ROOT', str(DEFAULT_DRIVE_BUNDLE_ROOT)))
LOCAL_BUNDLE_ROOT = Path(os.environ.get('SOKE_GEMMA_R256_LOCAL_BUNDLE_ROOT', str(DEFAULT_LOCAL_BUNDLE_ROOT)))
BASE_MODEL = os.environ.get('SOKE_GEMMA_R256_BASE_MODEL', 'google/gemma-3-270m')

RUN_ROOT_NAME = os.environ.get('SOKE_GEMMA_R256_RUN_ROOT_NAME', '07_soke_gemma3_r256a512_pretrain30_instruct5')
LOCAL_RUN_ROOT_BASE = LOCAL_BUNDLE_ROOT / RUN_ROOT_NAME / 'runs'
DRIVE_RUN_ROOT_BASE = DRIVE_BUNDLE_ROOT / RUN_ROOT_NAME / 'runs'
OUTPUT_ROOT = LOCAL_BUNDLE_ROOT / RUN_ROOT_NAME
DRIVE_OUTPUT_ROOT = DRIVE_BUNDLE_ROOT / RUN_ROOT_NAME

RUN_STAGES = [s.strip() for s in os.environ.get('SOKE_GEMMA_R256_RUN_STAGES', 'lm_pretrain,lm_instruct').split(',') if s.strip()]
EPOCHS_BY_STAGE = {
    'lm_pretrain': int(os.environ.get('SOKE_GEMMA_R256_PRETRAIN_EPOCHS', '30')),
    'lm_instruct': int(os.environ.get('SOKE_GEMMA_R256_INSTRUCT_EPOCHS', '5')),
}
if not RUN_STAGES:
    raise ValueError('SOKE_GEMMA_R256_RUN_STAGES must contain at least one stage.')
bad_stages = [stage for stage in RUN_STAGES if stage not in EPOCHS_BY_STAGE]
if bad_stages:
    raise ValueError(f'Unknown SOKE_GEMMA_R256_RUN_STAGES entries: {bad_stages}')

# Selected from the 06 rank sweep: lowest validation loss after 3 epochs.
LORA_R = int(os.environ.get('SOKE_GEMMA_R256_LORA_R', '256'))
LORA_ALPHA = int(os.environ.get('SOKE_GEMMA_R256_LORA_ALPHA', '512'))
LORA_DROPOUT = float(os.environ.get('SOKE_GEMMA_R256_LORA_DROPOUT', '0.05'))
LORA_TARGET_MODULES = os.environ.get('SOKE_GEMMA_R256_LORA_TARGET_MODULES', 'auto')

# Same effective batch as the successful sweep. Lower HARDWARE_BATCH_SIZE if memory is tight.
HARDWARE_BATCH_SIZE = int(os.environ.get('SOKE_GEMMA_R256_BATCH_SIZE', '16'))
VIRTUAL_BATCH_SIZE = int(os.environ.get('SOKE_GEMMA_R256_VIRTUAL_BATCH_SIZE', '128'))
GRAD_ACCUM = max(1, VIRTUAL_BATCH_SIZE // max(1, HARDWARE_BATCH_SIZE))
NUM_WORKERS = int(os.environ.get('SOKE_GEMMA_R256_NUM_WORKERS', '8'))
PIN_MEMORY = int(os.environ.get('SOKE_GEMMA_R256_PIN_MEMORY', '1'))
PERSISTENT_WORKERS = int(os.environ.get('SOKE_GEMMA_R256_PERSISTENT_WORKERS', '1'))
PREFETCH_FACTOR = int(os.environ.get('SOKE_GEMMA_R256_PREFETCH_FACTOR', '4'))
TF32 = int(os.environ.get('SOKE_GEMMA_R256_TF32', '1'))
TORCH_COMPILE = int(os.environ.get('SOKE_GEMMA_R256_TORCH_COMPILE', '0'))
TORCH_COMPILE_MODE = os.environ.get('SOKE_GEMMA_R256_TORCH_COMPILE_MODE', 'reduce-overhead')

MAX_SEQ_LEN = int(os.environ.get('SOKE_GEMMA_R256_MAX_SEQ_LEN', '1024'))
MAX_TRAIN_LOGICAL_ROWS = int(os.environ.get('SOKE_GEMMA_R256_MAX_TRAIN_LOGICAL_ROWS', '0'))
MAX_VAL_LOGICAL_ROWS = int(os.environ.get('SOKE_GEMMA_R256_MAX_VAL_LOGICAL_ROWS', '2048'))
EVAL_MAX_BATCHES = int(os.environ.get('SOKE_GEMMA_R256_EVAL_MAX_BATCHES', '0'))
VALIDATION_EVERY_EPOCHS = int(os.environ.get('SOKE_GEMMA_R256_VALIDATION_EVERY_EPOCHS', '1'))
MOTION_VAL_ENABLED = int(os.environ.get('SOKE_GEMMA_R256_MOTION_VAL_ENABLED', '0'))
MOTION_VAL_EVERY_EPOCHS = int(os.environ.get('SOKE_GEMMA_R256_MOTION_VAL_EVERY_EPOCHS', '5'))
MOTION_VAL_FRACTION = float(os.environ.get('SOKE_GEMMA_R256_MOTION_VAL_FRACTION', '0.1'))
MOTION_VAL_MAX_ROWS = int(os.environ.get('SOKE_GEMMA_R256_MOTION_VAL_MAX_ROWS', '256'))
MOTION_VAL_MAX_NEW_TOKENS = int(os.environ.get('SOKE_GEMMA_R256_MOTION_VAL_MAX_NEW_TOKENS', '384'))

LR = float(os.environ.get('SOKE_GEMMA_R256_LR', '2e-4'))
MOTION_TOKEN_LR = float(os.environ.get('SOKE_GEMMA_R256_MOTION_TOKEN_LR', '2e-5'))
ETA_MIN = float(os.environ.get('SOKE_GEMMA_R256_ETA_MIN', '1e-6'))
DTYPE = os.environ.get('SOKE_GEMMA_R256_DTYPE', 'bf16')
TRAIN_MOTION_TOKEN_EMBEDDINGS = int(os.environ.get('SOKE_GEMMA_R256_TRAIN_MOTION_TOKEN_EMBEDDINGS', '1'))
RANDOM_DROP = int(os.environ.get('SOKE_GEMMA_R256_RANDOM_DROP', '1'))
SEED = int(os.environ.get('SOKE_GEMMA_R256_SEED', '42'))

ATTN_IMPLEMENTATION = os.environ.get('SOKE_GEMMA_R256_ATTN_IMPLEMENTATION', 'auto')
SDPA_KERNEL = os.environ.get('SOKE_GEMMA_R256_SDPA_KERNEL', 'auto')
INSTALL_FLASH_ATTN_IF_MISSING = os.environ.get('SOKE_GEMMA_R256_INSTALL_FLASH_ATTN', '0') == '1'

RUN_TRAINING = os.environ.get('SOKE_GEMMA_R256_RUN_TRAINING', '1') == '1'
RESUME_TRAINING = os.environ.get('SOKE_GEMMA_R256_RESUME_TRAINING', '1') == '1'
FORCE_RERUN = os.environ.get('SOKE_GEMMA_R256_FORCE_RERUN', '0') == '1'
CONTINUE_ON_FAILURE = os.environ.get('SOKE_GEMMA_R256_CONTINUE_ON_FAILURE', '0') == '1'
REQUIRE_HF_TOKEN = os.environ.get('SOKE_GEMMA_R256_REQUIRE_HF_TOKEN', '1') == '1'
SYNC_TO_DRIVE = os.environ.get('SOKE_GEMMA_R256_SYNC_TO_DRIVE', '1') != '0'
DRIVE_SYNC_RETRIES = int(os.environ.get('SOKE_GEMMA_R256_DRIVE_SYNC_RETRIES', '5'))
DRIVE_SYNC_SLEEP_SEC = float(os.environ.get('SOKE_GEMMA_R256_DRIVE_SYNC_SLEEP_SEC', '20'))
DRIVE_SYNC_DELETE = int(os.environ.get('SOKE_GEMMA_R256_DRIVE_SYNC_DELETE', '1'))
SAVE_EVERY_EPOCHS = int(os.environ.get('SOKE_GEMMA_R256_SAVE_EVERY_EPOCHS', '1'))
KEEP_LOCAL_EPOCH_SAVES = int(os.environ.get('SOKE_GEMMA_R256_KEEP_LOCAL_EPOCH_SAVES', '3'))
KEEP_DRIVE_EPOCH_SAVES = int(os.environ.get('SOKE_GEMMA_R256_KEEP_DRIVE_EPOCH_SAVES', '5'))
INSTRUCT_INIT_ADAPTER_NAME = os.environ.get('SOKE_GEMMA_R256_INSTRUCT_INIT_ADAPTER_NAME', 'best_adapter')
AUTO_INIT_INSTRUCT_FROM_PRETRAIN = int(os.environ.get('SOKE_GEMMA_R256_AUTO_INIT_INSTRUCT_FROM_PRETRAIN', '1'))
RESUME_ADAPTER_NAME = os.environ.get('SOKE_GEMMA_R256_RESUME_ADAPTER_NAME', 'last_adapter')
LOG_EVERY_STEPS = int(os.environ.get('SOKE_GEMMA_R256_LOG_EVERY_STEPS', '25'))

print(json.dumps({
    'LOCAL_BUNDLE_ROOT': str(LOCAL_BUNDLE_ROOT),
    'DRIVE_BUNDLE_ROOT': str(DRIVE_BUNDLE_ROOT),
    'RUN_ROOT_NAME': RUN_ROOT_NAME,
    'RUN_STAGES': RUN_STAGES,
    'EPOCHS_BY_STAGE': EPOCHS_BY_STAGE,
    'BASE_MODEL': BASE_MODEL,
    'LORA_R': LORA_R,
    'LORA_ALPHA': LORA_ALPHA,
    'HARDWARE_BATCH_SIZE': HARDWARE_BATCH_SIZE,
    'GRAD_ACCUM': GRAD_ACCUM,
    'VIRTUAL_BATCH_SIZE': HARDWARE_BATCH_SIZE * GRAD_ACCUM,
    'VALIDATION_EVERY_EPOCHS': VALIDATION_EVERY_EPOCHS,
    'MOTION_VAL_ENABLED': MOTION_VAL_ENABLED,
    'ATTN_IMPLEMENTATION': ATTN_IMPLEMENTATION,
    'SDPA_KERNEL': SDPA_KERNEL,
    'RUN_TRAINING': RUN_TRAINING,
    'RESUME_TRAINING': RESUME_TRAINING,
    'SYNC_TO_DRIVE': SYNC_TO_DRIVE,
    'HF_TOKEN_PRESENT': bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')),
}, indent=2))


In [ ]:
# Mount Drive if needed, then copy scripts, instructions, and prebuilt SOKE motion-code JSONL data to local Colab storage.
import time


def run(cmd, *, check=True, env=None, retries=1, sleep_sec=5):
    printable = ' '.join(shlex.quote(str(x)) for x in cmd)
    last = None
    for attempt in range(1, int(retries) + 1):
        print(f'[{attempt}/{retries}] {printable}', flush=True)
        last = subprocess.run([str(x) for x in cmd], check=False, env=env)
        if last.returncode == 0:
            return last
        if attempt < int(retries):
            time.sleep(float(sleep_sec))
    if check:
        raise subprocess.CalledProcessError(last.returncode, [str(x) for x in cmd])
    return last


def mount_drive_if_needed():
    if not IN_COLAB:
        return
    if Path('/content/drive/MyDrive').exists():
        return
    from google.colab import drive
    drive.mount('/content/drive')


def resolve_drive_bundle_root():
    global DRIVE_BUNDLE_ROOT, DRIVE_OUTPUT_ROOT, DRIVE_RUN_ROOT_BASE
    if not IN_COLAB:
        return DRIVE_BUNDLE_ROOT
    mount_drive_if_needed()
    candidates = []
    for candidate in [
        DRIVE_BUNDLE_ROOT,
        Path('/content/drive/MyDrive/folder/COLAB/Tokenizer/04_soke_gemma3_270m_lora_lm'),
        Path('/content/drive/My Drive/folder/COLAB/Tokenizer/04_soke_gemma3_270m_lora_lm'),
    ]:
        if candidate not in candidates:
            candidates.append(candidate)
    for candidate in candidates:
        if candidate.exists():
            DRIVE_BUNDLE_ROOT = candidate
            DRIVE_OUTPUT_ROOT = DRIVE_BUNDLE_ROOT / RUN_ROOT_NAME
            DRIVE_RUN_ROOT_BASE = DRIVE_OUTPUT_ROOT / 'runs'
            return candidate
    parent = Path('/content/drive/MyDrive/folder/COLAB/Tokenizer')
    if parent.exists():
        print('Available Tokenizer dirs:', sorted(p.name for p in parent.iterdir() if p.is_dir())[:50])
    raise FileNotFoundError('Missing Drive bundle. Tried: ' + ', '.join(str(x) for x in candidates))


LOCAL_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
drive_root = resolve_drive_bundle_root() if IN_COLAB else DRIVE_BUNDLE_ROOT

if drive_root.exists() and drive_root.resolve() != LOCAL_BUNDLE_ROOT.resolve():
    for name in ['scripts', 'instructions']:
        src = drive_root / name
        dst = LOCAL_BUNDLE_ROOT / name
        if not src.exists():
            raise FileNotFoundError(f'Missing Drive folder: {src}')
        dst.mkdir(parents=True, exist_ok=True)
        run(['rsync', '-a', '--delete', str(src) + '/', str(dst) + '/'], retries=3)

    code_src = drive_root / 'outputs' / 'soke_motion_codes'
    code_dst = LOCAL_BUNDLE_ROOT / 'outputs' / 'soke_motion_codes'
    if not code_src.exists():
        raise FileNotFoundError(f'Missing prebuilt motion-code folder on Drive: {code_src}')
    code_dst.mkdir(parents=True, exist_ok=True)
    run(['rsync', '-a', '--partial', '--append-verify', str(code_src) + '/', str(code_dst) + '/'], retries=3, sleep_sec=10)
elif not (LOCAL_BUNDLE_ROOT / 'scripts').exists():
    raise FileNotFoundError(f'Missing local bundle scripts and Drive bundle was not available: {LOCAL_BUNDLE_ROOT}')

code_root = LOCAL_BUNDLE_ROOT / 'outputs' / 'soke_motion_codes'
for name in ['train_soke_motion_codes.jsonl', 'val_soke_motion_codes.jsonl', 'part_codecs.json']:
    path = code_root / name
    if not path.exists():
        raise FileNotFoundError(f'Missing required motion-code artifact: {path}')
    size = path.stat().st_size
    if size <= 0:
        raise RuntimeError(f'Empty required artifact, delete local copy and rerun this cell: {path}')
    print(path, size)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive bundle root:', drive_root)
print('Output root:', OUTPUT_ROOT)


In [ ]:
# Validate Colab dependencies, authenticate Hugging Face, then verify the selected attention backend.
from getpass import getpass
from importlib import metadata as importlib_metadata
import re
import torch


def _version_key(version):
    match = re.match(r'(\d+)\.(\d+)\.(\d+)', str(version))
    if not match:
        return (0, 0, 0)
    return tuple(int(x) for x in match.groups())


def ensure_torchao_compatible():
    minimum = '0.16.0'
    try:
        current = importlib_metadata.version('torchao')
    except importlib_metadata.PackageNotFoundError:
        print({'torchao_status': 'not_installed', 'action': 'none'})
        return
    if _version_key(current) < _version_key(minimum):
        print({'torchao_status': 'upgrade_required', 'current': current, 'minimum': minimum})
        run([sys.executable, '-m', 'pip', 'install', '-U', f'torchao>={minimum}'])
        current = importlib_metadata.version('torchao')
    if _version_key(current) < _version_key(minimum):
        raise RuntimeError(f'torchao upgrade failed: found {current}, need >= {minimum}')
    print({'torchao_status': 'ok', 'version': current})


def _get_colab_secret(name):
    if not IN_COLAB:
        return None
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value if value else None
    except Exception:
        return None


def ensure_hf_access():
    token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
    if not token:
        token = _get_colab_secret('HF_TOKEN') or _get_colab_secret('HUGGING_FACE_HUB_TOKEN')
        if token:
            os.environ['HF_TOKEN'] = token
            os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    if REQUIRE_HF_TOKEN and not token:
        token = getpass('HF token for gated Gemma model (input hidden): ').strip()
        if token:
            os.environ['HF_TOKEN'] = token
            os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    if REQUIRE_HF_TOKEN and not token:
        raise RuntimeError('Missing HF_TOKEN/HUGGING_FACE_HUB_TOKEN. Add a Colab secret named HF_TOKEN or paste a token when prompted.')
    if token:
        try:
            from huggingface_hub import login
            login(token=token, add_to_git_credential=False)
        except Exception as exc:
            raise RuntimeError('Hugging Face login failed. Check that the token is valid and has accepted Gemma access.') from exc
    try:
        from transformers import AutoTokenizer
        AutoTokenizer.from_pretrained(BASE_MODEL, token=token if token else None)
    except Exception as exc:
        raise RuntimeError(f'Could not access {BASE_MODEL}. Check HF token and model license access before starting training.') from exc
    print({'base_model_access': 'ok', 'base_model': BASE_MODEL, 'hf_token_present': bool(token)})


ensure_torchao_compatible()
ensure_hf_access()

if not torch.cuda.is_available():
    raise RuntimeError('Training requires a CUDA GPU runtime.')

if ATTN_IMPLEMENTATION == 'auto':
    try:
        import flash_attn
        from transformers.utils import is_flash_attn_2_available
        has_fa2 = bool(is_flash_attn_2_available())
    except Exception:
        flash_attn = None
        has_fa2 = False
    print({
        'torch': torch.__version__,
        'cuda_device': torch.cuda.get_device_name(0),
        'attn_implementation': ATTN_IMPLEMENTATION,
        'resolved_by_trainer': 'flash_attention_2' if has_fa2 else 'sdpa',
        'sdpa_kernel_if_needed': SDPA_KERNEL,
        'flash_attn_version': getattr(flash_attn, '__version__', '') if flash_attn is not None else '',
        'flash_attn_2_available': has_fa2,
    })
elif ATTN_IMPLEMENTATION == 'flash_attention_2':
    try:
        import flash_attn
    except Exception as exc:
        if not INSTALL_FLASH_ATTN_IF_MISSING:
            raise RuntimeError('flash-attn is missing. Install it manually or use ATTN_IMPLEMENTATION=auto.') from exc
        run([sys.executable, '-m', 'pip', 'install', '-U', 'ninja', 'packaging', 'psutil'])
        env = os.environ.copy()
        env.setdefault('MAX_JOBS', '4')
        run([sys.executable, '-m', 'pip', 'install', '-U', 'flash-attn', '--no-build-isolation'], env=env)
        import flash_attn
    from transformers.utils import is_flash_attn_2_available
    if not is_flash_attn_2_available():
        raise RuntimeError('Transformers reports FlashAttention-2 is unavailable after flash-attn import.')
    print({'torch': torch.__version__, 'cuda_device': torch.cuda.get_device_name(0), 'attn_implementation': ATTN_IMPLEMENTATION, 'flash_attn_version': getattr(flash_attn, '__version__', 'unknown')})
else:
    print({'torch': torch.__version__, 'cuda_device': torch.cuda.get_device_name(0), 'attn_implementation': ATTN_IMPLEMENTATION, 'sdpa_kernel': SDPA_KERNEL})


In [ ]:
# Optional quick dataset inspection before training.
INSPECT_N = int(os.environ.get('SOKE_GEMMA_R256_INSPECT_N', '0'))
if INSPECT_N > 0:
    cmd = [
        sys.executable, LOCAL_BUNDLE_ROOT / 'scripts' / 'inspect_soke_gemma_dataset.py',
        '--code-root', code_root,
        '--instructions-root', LOCAL_BUNDLE_ROOT / 'instructions',
        '--split', 'train',
        '--stage', RUN_STAGES[0],
        '--n', INSPECT_N,
        '--seed', SEED,
        '--tokenizer', BASE_MODEL,
    ]
    run(cmd)
else:
    print('Inspection skipped because SOKE_GEMMA_R256_INSPECT_N=0')


In [ ]:
# Train/resume pretrain then instruction tuning.
def adapter_ready(path):
    path = Path(path)
    has_config = (path / 'adapter_config.json').exists()
    has_weights = (path / 'adapter_model.safetensors').exists() or (path / 'adapter_model.bin').exists()
    return has_config and has_weights


def stage_epoch(path):
    path = Path(path)
    hist_path = path / 'history.csv'
    if hist_path.exists():
        try:
            import pandas as pd
            hist = pd.read_csv(hist_path)
            if not hist.empty and 'epoch' in hist.columns:
                return int(hist['epoch'].max())
        except Exception:
            pass
    latest = path / 'latest_status.json'
    if latest.exists():
        try:
            payload = json.loads(latest.read_text(encoding='utf-8'))
            return int(payload.get('epoch') or payload.get('resume_epoch') or 0)
        except Exception:
            pass
    final = path / 'final_status.json'
    if final.exists():
        try:
            payload = json.loads(final.read_text(encoding='utf-8'))
            return int(payload.get('epochs') or payload.get('target_epochs') or 0)
        except Exception:
            pass
    return 0


def stage_resume_ready(path):
    path = Path(path)
    return adapter_ready(path / RESUME_ADAPTER_NAME) and (path / 'last_train_state.pt').exists()


def sync_stage_from_drive_if_newer(local_run_root, drive_run_root):
    if not (SYNC_TO_DRIVE and drive_run_root.exists()):
        return False
    drive_epoch = stage_epoch(drive_run_root)
    local_epoch = stage_epoch(local_run_root)
    if drive_epoch <= 0:
        return False
    if (not stage_resume_ready(local_run_root)) or drive_epoch > local_epoch:
        print(f'Restoring {drive_run_root.name} from Drive epoch {drive_epoch} -> local epoch {local_epoch}', flush=True)
        local_run_root.mkdir(parents=True, exist_ok=True)
        run(['rsync', '-a', '--partial', str(drive_run_root) + '/', str(local_run_root) + '/'], retries=3, sleep_sec=10)
        return True
    return False


def tee_training_command(cmd, log_path, *, env=None):
    printable = ' '.join(shlex.quote(str(x)) for x in cmd)
    print('[train]', printable, flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = []
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n' + '=' * 120 + '\n')
        log.write(printable + '\n')
        log.flush()
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            tail.append(line.rstrip('\n'))
            if len(tail) > 160:
                tail = tail[-160:]
        returncode = proc.wait()
    return subprocess.CompletedProcess([str(x) for x in cmd], returncode), tail


train_script = LOCAL_BUNDLE_ROOT / 'scripts' / 'train_gemma_soke_lora.py'
if not train_script.exists():
    raise FileNotFoundError(train_script)

stage_status = []
previous_pretrain_adapter = None
for stage_idx, stage in enumerate(RUN_STAGES):
    stage_epochs = int(EPOCHS_BY_STAGE[stage])
    local_run_root = LOCAL_RUN_ROOT_BASE / stage
    drive_run_root = DRIVE_RUN_ROOT_BASE / stage
    if FORCE_RERUN and local_run_root.exists():
        shutil.rmtree(local_run_root)
    restored = sync_stage_from_drive_if_newer(local_run_root, drive_run_root)
    current_epoch = stage_epoch(local_run_root)
    resume_current_stage = bool(RESUME_TRAINING and stage_resume_ready(local_run_root) and current_epoch < stage_epochs)
    if current_epoch >= stage_epochs and stage_resume_ready(local_run_root) and not FORCE_RERUN:
        print(f'Skipping {stage}; already complete at epoch {current_epoch}/{stage_epochs}')
        if stage == 'lm_pretrain':
            candidate = local_run_root / INSTRUCT_INIT_ADAPTER_NAME
            previous_pretrain_adapter = candidate if adapter_ready(candidate) else local_run_root / RESUME_ADAPTER_NAME
        stage_status.append({'stage': stage, 'skipped': True, 'current_epoch': current_epoch, 'target_epochs': stage_epochs, 'restored_from_drive': restored})
        continue

    init_adapter = None
    if stage == 'lm_instruct' and AUTO_INIT_INSTRUCT_FROM_PRETRAIN and not resume_current_stage:
        candidate = previous_pretrain_adapter or (LOCAL_RUN_ROOT_BASE / 'lm_pretrain' / INSTRUCT_INIT_ADAPTER_NAME)
        if not adapter_ready(candidate):
            drive_candidate = DRIVE_RUN_ROOT_BASE / 'lm_pretrain' / INSTRUCT_INIT_ADAPTER_NAME
            if adapter_ready(drive_candidate):
                candidate.parent.mkdir(parents=True, exist_ok=True)
                run(['rsync', '-a', str(drive_candidate) + '/', str(candidate) + '/'], retries=3, sleep_sec=10)
        if adapter_ready(candidate):
            init_adapter = candidate
            print('Initializing lm_instruct from:', init_adapter)
        else:
            raise FileNotFoundError(f'No valid pretrain adapter found for lm_instruct. Tried local/Drive {INSTRUCT_INIT_ADAPTER_NAME}.')

    cmd = [
        sys.executable, train_script,
        '--code-root', code_root,
        '--instructions-root', LOCAL_BUNDLE_ROOT / 'instructions',
        '--run-root', local_run_root,
        '--base-model', BASE_MODEL,
        '--stage', stage,
        '--epochs', stage_epochs,
        '--resume-training', int(resume_current_stage),
        '--resume-adapter-name', RESUME_ADAPTER_NAME,
        '--batch-size', HARDWARE_BATCH_SIZE,
        '--grad-accum', GRAD_ACCUM,
        '--num-workers', NUM_WORKERS,
        '--pin-memory', PIN_MEMORY,
        '--persistent-workers', PERSISTENT_WORKERS,
        '--prefetch-factor', PREFETCH_FACTOR,
        '--tf32', TF32,
        '--torch-compile', TORCH_COMPILE,
        '--torch-compile-mode', TORCH_COMPILE_MODE,
        '--max-seq-len', MAX_SEQ_LEN,
        '--gradient-checkpointing', '1',
        '--attn-implementation', ATTN_IMPLEMENTATION,
        '--sdpa-kernel', SDPA_KERNEL,
        '--max-train-logical-rows', MAX_TRAIN_LOGICAL_ROWS,
        '--max-val-logical-rows', MAX_VAL_LOGICAL_ROWS,
        '--eval-max-batches', EVAL_MAX_BATCHES,
        '--validation-every-epochs', VALIDATION_EVERY_EPOCHS,
        '--motion-val-enabled', MOTION_VAL_ENABLED,
        '--motion-val-every-epochs', MOTION_VAL_EVERY_EPOCHS,
        '--motion-val-fraction', MOTION_VAL_FRACTION,
        '--motion-val-max-rows', MOTION_VAL_MAX_ROWS,
        '--motion-val-max-new-tokens', MOTION_VAL_MAX_NEW_TOKENS,
        '--lr', LR,
        '--motion-token-lr', MOTION_TOKEN_LR,
        '--eta-min', ETA_MIN,
        '--dtype', DTYPE,
        '--lora-r', LORA_R,
        '--lora-alpha', LORA_ALPHA,
        '--lora-dropout', LORA_DROPOUT,
        '--lora-target-modules', LORA_TARGET_MODULES,
        '--train-motion-token-embeddings', TRAIN_MOTION_TOKEN_EMBEDDINGS,
        '--random-drop', RANDOM_DROP,
        '--seed', SEED + stage_idx,
        '--log-every-steps', LOG_EVERY_STEPS,
        '--save-every-epochs', SAVE_EVERY_EPOCHS,
        '--keep-local-epoch-saves', KEEP_LOCAL_EPOCH_SAVES,
        '--sync-to-drive', int(SYNC_TO_DRIVE and DRIVE_BUNDLE_ROOT.exists()),
        '--drive-sync-dest', drive_run_root,
        '--drive-sync-retries', DRIVE_SYNC_RETRIES,
        '--drive-sync-sleep-sec', DRIVE_SYNC_SLEEP_SEC,
        '--drive-sync-delete', DRIVE_SYNC_DELETE,
        '--keep-drive-epoch-saves', KEEP_DRIVE_EPOCH_SAVES,
    ]
    if init_adapter is not None:
        cmd += ['--init-adapter', init_adapter]
    env = os.environ.copy()
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    if not RUN_TRAINING:
        print('Training disabled. Would run:', ' '.join(shlex.quote(str(x)) for x in cmd))
        continue
    result, tail = tee_training_command(cmd, local_run_root / 'subprocess_stdout_stderr.log', env=env)
    row = {
        'stage': stage,
        'returncode': result.returncode,
        'target_epochs': stage_epochs,
        'current_epoch_before_run': current_epoch,
        'restored_from_drive': restored,
        'local_run_root': str(local_run_root),
        'drive_run_root': str(drive_run_root),
        'error_tail': tail[-60:] if result.returncode != 0 else [],
    }
    stage_status.append(row)
    (OUTPUT_ROOT / 'stage_status.json').write_text(json.dumps(stage_status, indent=2), encoding='utf-8')
    if result.returncode != 0:
        if SYNC_TO_DRIVE and DRIVE_BUNDLE_ROOT.exists():
            DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
            run(['rsync', '-a', str(OUTPUT_ROOT / 'stage_status.json'), str(DRIVE_OUTPUT_ROOT / 'stage_status.json')], check=False, retries=3, sleep_sec=10)
            if local_run_root.exists():
                run(['rsync', '-a', str(local_run_root) + '/', str(drive_run_root) + '/'], check=False, retries=3, sleep_sec=10)
        message = f"Stage {stage} failed with returncode {result.returncode}. Log: {local_run_root / 'subprocess_stdout_stderr.log'}\n" + '\n'.join(tail[-80:])
        if CONTINUE_ON_FAILURE:
            print(message, flush=True)
            continue
        raise RuntimeError(message)
    if stage == 'lm_pretrain':
        candidate = local_run_root / INSTRUCT_INIT_ADAPTER_NAME
        previous_pretrain_adapter = candidate if adapter_ready(candidate) else local_run_root / RESUME_ADAPTER_NAME

(OUTPUT_ROOT / 'stage_status.json').write_text(json.dumps(stage_status, indent=2), encoding='utf-8')
stage_status


In [ ]:
# Summarize saved histories and plot train/validation curves.
import matplotlib.pyplot as plt
import pandas as pd

summary_rows = []
history_rows = []
for stage in ['lm_pretrain', 'lm_instruct']:
    run_root = LOCAL_RUN_ROOT_BASE / stage
    hist_path = run_root / 'history.csv'
    if not hist_path.exists():
        continue
    hist = pd.read_csv(hist_path)
    hist['stage'] = stage
    history_rows.append(hist)
    last = hist.tail(1).iloc[0].to_dict()
    summary_rows.append({
        'stage': stage,
        'epochs': int(last.get('epoch', 0)),
        'global_step': int(last.get('global_step', 0)),
        'train_loss_lower_better': float(last.get('train_loss', float('nan'))),
        'val_loss_lower_better': float(last.get('val_loss', float('nan'))),
        'val_token_accuracy_higher_better': float(last.get('val_token_accuracy_higher_better', float('nan'))),
        'lr': float(last.get('lr', float('nan'))),
        'run_root': str(run_root),
    })

summary_df = pd.DataFrame(summary_rows)
history_df = pd.concat(history_rows, ignore_index=True) if history_rows else pd.DataFrame()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
plots_root = OUTPUT_ROOT / 'plots'
plots_root.mkdir(parents=True, exist_ok=True)
summary_csv = OUTPUT_ROOT / 'r256a512_stage_summary.csv'
history_csv = OUTPUT_ROOT / 'r256a512_stage_history.csv'
summary_df.to_csv(summary_csv, index=False)
history_df.to_csv(history_csv, index=False)

if not history_df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    for stage, group in history_df.groupby('stage', sort=False):
        ax.plot(group['epoch'], group['train_loss'], marker='o', label=f'{stage} train loss')
        ax.plot(group['epoch'], group['val_loss'], marker='x', linestyle='--', label=f'{stage} val loss')
    ax.set_title('Gemma SOKE r256 alpha512 Training History (lower better)')
    ax.set_xlabel('Epoch within stage')
    ax.set_ylabel('Loss (lower better)')
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    plot_path = plots_root / 'r256a512_stage_loss.png'
    fig.savefig(plot_path, dpi=180)
    plt.show()
else:
    plot_path = plots_root / 'r256a512_stage_loss.png'
    print('No history found yet.')

print('Wrote:', summary_csv)
print('Wrote:', history_csv)
print('Plot:', plot_path)
display(summary_df)


In [ ]:
# Sync notebook-level summary artifacts to Drive.
if SYNC_TO_DRIVE and DRIVE_BUNDLE_ROOT.exists():
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    run(['rsync', '-a', str(OUTPUT_ROOT) + '/', str(DRIVE_OUTPUT_ROOT) + '/'], retries=DRIVE_SYNC_RETRIES, sleep_sec=DRIVE_SYNC_SLEEP_SEC)
    print('Synced:', DRIVE_OUTPUT_ROOT)
else:
    print('Drive sync skipped.')


In [ ]:
# Optional Colab close/unassign-runtime cell. Disable with SOKE_GEMMA_R256_UNASSIGN_RUNTIME=0.
UNASSIGN_RUNTIME = os.environ.get('SOKE_GEMMA_R256_UNASSIGN_RUNTIME', '1') == '1'
if IN_COLAB and UNASSIGN_RUNTIME:
    try:
        from google.colab import runtime
        print('Unassigning Colab runtime now.')
        runtime.unassign()
    except Exception as exc:
        print('Could not unassign runtime automatically:', repr(exc))
else:
    print('Runtime left active.')
